In [ ]:
# Credits and Acknowledgements
# The code in this notebook were adapted from https://github.com/yfukai/laptrack

# Tracking 3D object by their centroids

In [1]:
%pip install -q --upgrade numpy pandas laptrack napari[all] scikit-image jupyter

## Importing packages

`laptrack.LapTrack` is the core object for tracking.

In [2]:
import napari
import numpy as np
import pandas as pd
from skimage.measure import regionprops_table
from matplotlib import pyplot as plt

from laptrack import LapTrack
from laptrack import datasets
from laptrack import data_conversion

## Loading images and showing them in napari

In [ ]:
viewer = napari.Viewer(ndisplay=3)
images, labels = datasets.HL60_3D_synthesized()
viewer.add_image(images)
viewer.add_labels(labels)

In [ ]:
images, labels = datasets.HL60_3D_synthesized()

# Calculate centroids of the labels

In [ ]:
dfs = []
for frame, l in enumerate(labels):
    dfs.append(
        pd.DataFrame(regionprops_table(l, properties=["label", "centroid"])).assign(
            frame=frame
        )
    )
regionprops_df = pd.concat(dfs, ignore_index=True)
regionprops_df.rename(
    columns={"centroid-0": "z", "centroid-1": "y", "centroid-2": "x"}, inplace=True
)
regionprops_df.head()

In [ ]:
viewer.add_points(
    regionprops_df[["frame", "z", "y", "x"]],
    size=2,
    name="centroid",
    blending="additive",
)

# Track centroid

First, initialize the `LapTrack` object with parameters.

In [5]:
max_distance = 15
lt = LapTrack(
    track_dist_metric="sqeuclidean",  # The similarity metric for particles. See `scipy.spatial.distance.cdist` for allowed values.
    splitting_dist_metric="sqeuclidean",  # The similarity metric for splits.
    # the square of the cutoff distance for the "sqeuclidean" metric
    track_cost_cutoff=max_distance**2,
    gap_closing_cost_cutoff=max_distance**2,
    splitting_cost_cutoff=max_distance**2,
)

In [6]:
track_df, split_df, merge_df = lt.predict_dataframe(
    regionprops_df,
    coordinate_cols=["z", "y", "x"],  # the column names for the coordinates
    frame_col="frame",  # the column name for the frame (default "frame")
    only_coordinate_cols=False,
)
track_df = track_df.reset_index()

- track_id : unique ID for each track
- tree_id : unique ID for each "clonal" set of track (track sharing the same ancestor)

In [ ]:
track_df

In [ ]:
split_df

In [ ]:
merge_df.head()

In [ ]:
graph = data_conversion.convert_split_merge_df_to_napari_graph(split_df, merge_df)
viewer.add_tracks(track_df[["track_id", "frame", "z", "y", "x"]], graph=graph)

In [ ]:
viewer.dims.current_step = (74, 0, 0, 0)
plt.imshow(viewer.screenshot())
plt.xticks([])
plt.yticks([])